In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🌾 EcoCrop Tunisia — Crop Yield Prediction\n",
    "### Complete Analysis Report\n",
    "\n",
    "**Authors:** Ferdaws Saidi & Aya Gharsalli  \n",
    "**Date:** 2024  \n",
    "**Objective:** Predict cereal crop yield (Tonnes) across Tunisia's 24 governorates using weather and regional data (2016–2024).\n",
    "\n",
    "---\n",
    "\n",
    "## Table of Contents\n",
    "1. Imports & Configuration\n",
    "2. Data Loading & First Look\n",
    "3. Descriptive Statistics\n",
    "4. Missing Values & Data Quality\n",
    "5. Weather Feature Distributions\n",
    "6. Crop Yield Distributions\n",
    "7. Outlier Detection (Boxplots)\n",
    "8. Production by Governorate\n",
    "9. Temporal Trends\n",
    "10. Correlation Analysis\n",
    "11. Feature Engineering\n",
    "12. Preprocessing (Encoding & Scaling)\n",
    "13. Model Training (Baseline)\n",
    "14. Model Evaluation & Comparison\n",
    "15. Cross-Validation\n",
    "16. Actual vs Predicted Plots\n",
    "17. Residual Analysis\n",
    "18. Feature Importance\n",
    "19. Hyperparameter Tuning (GridSearchCV)\n",
    "20. Final Model Evaluation\n",
    "21. Error Analysis by Governorate\n",
    "22. Conclusions & Recommendations"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Imports & Configuration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.ticker as mticker\n",
    "import seaborn as sns\n",
    "from scipy import stats\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "plt.rcParams['figure.dpi'] = 120\n",
    "plt.rcParams['font.family'] = 'DejaVu Sans'\n",
    "sns.set_theme(style='whitegrid', palette='muted')\n",
    "PALETTE = sns.color_palette('Set2')\n",
    "\n",
    "CROP_COLS = [\n",
    "    'Cereales (T)', 'Maraichage (T)', 'Legumineuses (T)',\n",
    "    'Fourrages (T)', 'Arboriculture (T)', 'Olives (T)',\n",
    "    'Cultures industrielles (T)'\n",
    "]\n",
    "WEATHER_COLS = [\n",
    "    'precipitation', 'hc_air_temperature',\n",
    "    'hc_relative_humidity', 'solar_radiation', 'wind_speed_sonic'\n",
    "]\n",
    "\n",
    "print('✅ Libraries loaded successfully.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Data Loading & First Look"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df = pd.read_csv('ecocrop_cleaned_data (1).csv')\n",
    "\n",
    "print(f'📐 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')\n",
    "print(f'📅 Years covered: {df[\"Year\"].min()} → {df[\"Year\"].max()}')\n",
    "print(f'🗺️  Governorates: {df[\"Governorate\"].nunique()} unique regions')\n",
    "print(f'🔁 Duplicates: {df.duplicated().sum()}')\n",
    "print(f'❌ Missing values: {df.isnull().sum().sum()}')\n",
    "print('\\n--- Column types ---')\n",
    "print(df.dtypes)\n",
    "print('\\n--- First 5 rows ---')\n",
    "display(df.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Descriptive Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('=== WEATHER VARIABLES ===')\n",
    "display(df[WEATHER_COLS].describe().round(2))\n",
    "\n",
    "print('\\n=== CROP YIELD VARIABLES (Tonnes) ===')\n",
    "display(df[CROP_COLS].describe().round(2))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Missing Values & Data Quality"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "missing = df.isnull().sum()\n",
    "missing_pct = (missing / len(df) * 100).round(2)\n",
    "missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})\n",
    "missing_df = missing_df[missing_df['Missing Count'] > 0]\n",
    "\n",
    "if missing_df.empty:\n",
    "    print('✅ No missing values found in the dataset!')\n",
    "else:\n",
    "    print(missing_df)\n",
    "\n",
    "print(f'\\n🔁 Duplicate rows: {df.duplicated().sum()}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Weather Feature Distributions"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 3, figsize=(16, 9))\n",
    "axes = axes.flatten()\n",
    "\n",
    "weather_labels = [\n",
    "    'Precipitation (mm)', 'Air Temperature (°C)',\n",
    "    'Relative Humidity (%)', 'Solar Radiation (W/m²)',\n",
    "    'Wind Speed (m/s)'\n",
    "]\n",
    "\n",
    "for i, (col, label) in enumerate(zip(WEATHER_COLS, weather_labels)):\n",
    "    sns.histplot(df[col], kde=True, ax=axes[i], color=PALETTE[i], bins=35)\n",
    "    axes[i].set_title(f'Distribution of {label}', fontsize=11)\n",
    "    axes[i].set_xlabel(label)\n",
    "    axes[i].set_ylabel('Frequency')\n",
    "    mean_val = df[col].mean()\n",
    "    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.2,\n",
    "                    label=f'Mean: {mean_val:.1f}')\n",
    "    axes[i].legend(fontsize=8)\n",
    "\n",
    "axes[-1].set_visible(False)\n",
    "fig.suptitle('🌤️ Weather Feature Distributions — EcoCrop Tunisia',\n",
    "             fontsize=14, fontweight='bold', y=1.01)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Crop Yield Distributions"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 4, figsize=(18, 9))\n",
    "axes = axes.flatten()\n",
    "\n",
    "crop_labels = ['Cereals', 'Market Gardening', 'Legumes',\n",
    "               'Fodder', 'Arboriculture', 'Olives', 'Industrial Crops']\n",
    "\n",
    "for i, (col, label) in enumerate(zip(CROP_COLS, crop_labels)):\n",
    "    data_nz = df[col][df[col] > 0]\n",
    "    sns.histplot(data_nz, kde=True, ax=axes[i],\n",
    "                 color=PALETTE[i % len(PALETTE)], bins=30)\n",
    "    axes[i].set_title(f'{label}\\n(non-zero, n={len(data_nz):,})', fontsize=9)\n",
    "    axes[i].set_xlabel('Tonnes')\n",
    "    axes[i].set_ylabel('Freq')\n",
    "    axes[i].xaxis.set_major_formatter(\n",
    "        mticker.FuncFormatter(lambda x, p: f'{x/1e3:.0f}k'))\n",
    "\n",
    "axes[-1].set_visible(False)\n",
    "fig.suptitle('🌾 Crop Yield Distributions (Non-Zero, Tonnes)',\n",
    "             fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print('Skewness of crop yields:')\n",
    "print(df[CROP_COLS].skew().round(3))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Outlier Detection (Boxplots)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 5, figsize=(18, 5))\n",
    "\n",
    "for i, (col, label) in enumerate(zip(WEATHER_COLS, weather_labels)):\n",
    "    sns.boxplot(y=df[col], ax=axes[i], color=PALETTE[i], width=0.5, fliersize=3)\n",
    "    axes[i].set_title(label, fontsize=9)\n",
    "\n",
    "fig.suptitle('📦 Boxplots — Weather Feature Outliers', fontsize=13, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print('Outlier counts (IQR method):')\n",
    "for col in WEATHER_COLS:\n",
    "    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)\n",
    "    IQR = Q3 - Q1\n",
    "    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]\n",
    "    print(f'  {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Production by Governorate"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df['Total_Production'] = df[CROP_COLS].sum(axis=1)\n",
    "gov_prod = df.groupby('Governorate')['Total_Production'].sum().sort_values(ascending=False)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(14, 6))\n",
    "colors = sns.color_palette('viridis', len(gov_prod))\n",
    "bars = ax.bar(gov_prod.index, gov_prod.values / 1e6, color=colors)\n",
    "ax.set_title('🗺️ Total Agricultural Production by Governorate',\n",
    "             fontsize=13, fontweight='bold')\n",
    "ax.set_ylabel('Total Production (Million Tonnes)')\n",
    "plt.xticks(rotation=45, ha='right', fontsize=8)\n",
    "ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:.1f}M'))\n",
    "\n",
    "for bar, val in zip(bars[:5], gov_prod.values[:5]):\n",
    "    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,\n",
    "            f'{val/1e6:.1f}M', ha='center', va='bottom', fontsize=7, fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Temporal Trends"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "yearly = df.groupby('Year')[CROP_COLS].mean()\n",
    "\n",
    "fig, axes = plt.subplots(2, 4, figsize=(18, 9))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, (col, label) in enumerate(zip(CROP_COLS, crop_labels)):\n",
    "    axes[i].plot(yearly.index, yearly[col], marker='o',\n",
    "                 color=PALETTE[i % len(PALETTE)], linewidth=2, markersize=5)\n",
    "    axes[i].fill_between(yearly.index, yearly[col], alpha=0.15,\n",
    "                         color=PALETTE[i % len(PALETTE)])\n",
    "    axes[i].set_title(f'{label}', fontsize=10)\n",
    "    axes[i].set_xlabel('Year')\n",
    "    axes[i].set_ylabel('Avg Tonnes')\n",
    "    axes[i].yaxis.set_major_formatter(\n",
    "        mticker.FuncFormatter(lambda x, p: f'{x/1e3:.0f}k'))\n",
    "\n",
    "axes[-1].set_visible(False)\n",
    "fig.suptitle('📅 Average Crop Yield per Year (2016–2024)',\n",
    "             fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Correlation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "all_num_cols = WEATHER_COLS + CROP_COLS + ['Year']\n",
    "corr_matrix = df[all_num_cols].corr()\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(14, 10))\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',\n",
    "            cmap='RdYlGn', center=0, linewidths=0.5,\n",
    "            annot_kws={'size': 7}, ax=ax, square=True)\n",
    "ax.set_title('🔗 Full Correlation Matrix — Weather × Crops × Year',\n",
    "             fontsize=13, fontweight='bold')\n",
    "plt.xticks(rotation=45, ha='right', fontsize=8)\n",
    "plt.yticks(fontsize=8)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Feature Engineering"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def temp_to_season(t):\n",
    "    if t < 10: return 'Winter'\n",
    "    elif t < 18: return 'Spring/Autumn'\n",
    "    else: return 'Summer'\n",
    "\n",
    "df['Season'] = df['hc_air_temperature'].apply(temp_to_season)\n",
    "\n",
    "df['Rainfall_Bin'] = pd.cut(\n",
    "    df['precipitation'],\n",
    "    bins=[-0.01, 0.0, 2.0, 10.0, 30.0, 200.0],\n",
    "    labels=['No Rain', 'Trace', 'Light', 'Moderate', 'Heavy']\n",
    ")\n",
    "\n",
    "df['Temp_Zone'] = pd.cut(\n",
    "    df['hc_air_temperature'],\n",
    "    bins=[0, 10, 18, 25, 45],\n",
    "    labels=['Cold', 'Mild', 'Warm', 'Hot']\n",
    ")\n",
    "\n",
    "df['Heat_Index'] = df['hc_air_temperature'] * (1 + df['hc_relative_humidity'] / 100)\n",
    "\n",
    "print('✅ Engineered features created:')\n",
    "print(df[['Season', 'Rainfall_Bin', 'Temp_Zone', 'Heat_Index']].head(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 12. Preprocessing (Encoding & Scaling)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.preprocessing import LabelEncoder, StandardScaler\n",
    "from sklearn.model_selection import train_test_split\n",
    "\n",
    "# Encode categoricals\n",
    "le_gov = LabelEncoder()\n",
    "df['Governorate_Encoded'] = le_gov.fit_transform(df['Governorate'])\n",
    "\n",
    "le_season = LabelEncoder()\n",
    "df['Season_Encoded'] = le_season.fit_transform(df['Season'])\n",
    "\n",
    "le_rain = LabelEncoder()\n",
    "df['Rainfall_Bin_Encoded'] = le_rain.fit_transform(df['Rainfall_Bin'].astype(str))\n",
    "\n",
    "le_temp = LabelEncoder()\n",
    "df['Temp_Zone_Encoded'] = le_temp.fit_transform(df['Temp_Zone'].astype(str))\n",
    "\n",
    "# Define features and target\n",
    "X_COLS = [\n",
    "    'precipitation', 'hc_air_temperature', 'hc_relative_humidity',\n",
    "    'solar_radiation', 'wind_speed_sonic', 'Year',\n",
    "    'Governorate_Encoded', 'Season_Encoded',\n",
    "    'Rainfall_Bin_Encoded', 'Temp_Zone_Encoded', 'Heat_Index'\n",
    "]\n",
    "TARGET = 'Cereales (T)'\n",
    "\n",
    "X = df[X_COLS]\n",
    "y = df[TARGET]\n",
    "\n",
    "# Train/test split\n",
    "X_train_raw, X_test_raw, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "# Scale\n",
    "scaler = StandardScaler()\n",
    "X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=X_COLS)\n",
    "X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=X_COLS)\n",
    "\n",
    "print(f'✅ Train: {X_train.shape[0]:,} samples')\n",
    "print(f'✅ Test:  {X_test.shape[0]:,} samples')\n",
    "print(f'📌 Features: {len(X_COLS)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 13. Model Training (Baseline)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.linear_model import LinearRegression\n",
    "from sklearn.tree import DecisionTreeRegressor\n",
    "from sklearn.ensemble import RandomForestRegressor\n",
    "\n",
    "models = {\n",
    "    'Linear Regression': LinearRegression(),\n",
    "    'Decision Tree': DecisionTreeRegressor(random_state=42),\n",
    "    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)\n",
    "}\n",
    "\n",
    "trained = {}\n",
    "for name, model in models.items():\n",
    "    model.fit(X_train, y_train)\n",
    "    trained[name] = model\n",
    "    print(f'✅ {name} trained.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 14. Model Evaluation & Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error\n",
    "\n",
    "def evaluate_model(model, name, X_tr, y_tr, X_te, y_te):\n",
    "    preds_train = model.predict(X_tr)\n",
    "    preds_test = model.predict(X_te)\n",
    "    return {\n",
    "        'Model': name,\n",
    "        'R² Train': round(r2_score(y_tr, preds_train), 4),\n",
    "        'R² Test': round(r2_score(y_te, preds_test), 4),\n",
    "        'RMSE': round(np.sqrt(mean_squared_error(y_te, preds_test)), 2),\n",
    "        'MAE': round(mean_absolute_error(y_te, preds_test), 2)\n",
    "    }\n",
    "\n",
    "results = [evaluate_model(m, n, X_train, y_train, X_test, y_test)\n",
    "           for n, m in trained.items()]\n",
    "results_df = pd.DataFrame(results).set_index('Model')\n",
    "\n",
    "print('=== BASELINE MODEL COMPARISON ===')\n",
    "display(results_df.style.background_gradient(cmap='RdYlGn', subset=['R² Test'])\n",
    "         .format({'R² Train': '{:.4f}', 'R² Test': '{:.4f}',\n",
    "                  'RMSE': '{:,.0f}', 'MAE': '{:,.0f}'}))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Bar chart comparison\n",
    "fig, axes = plt.subplots(1, 3, figsize=(16, 5))\n",
    "model_names = results_df.index.tolist()\n",
    "colors_bar = [PALETTE[0], PALETTE[1], PALETTE[2]]\n",
    "\n",
    "for idx, (metric, ax_title) in enumerate([('R² Test', 'R² Score (Test)'),\n",
    "                                          ('RMSE', 'RMSE (lower=better)'),\n",
    "                                          ('MAE', 'MAE (lower=better)')]):\n",
    "    bars = axes[idx].bar(model_names, results_df[metric],\n",
    "                         color=colors_bar, edgecolor='black', linewidth=0.5)\n",
    "    axes[idx].set_title(ax_title, fontsize=12, fontweight='bold')\n",
    "    for bar, val in zip(bars, results_df[metric]):\n",
    "        axes[idx].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,\n",
    "                       f'{val:.4f}' if metric == 'R² Test' else f'{val/1e3:.1f}k',\n",
    "                       ha='center', fontsize=9)\n",
    "    plt.setp(axes[idx].xaxis.get_majorticklabels(), rotation=15)\n",
    "\n",
    "fig.suptitle('📊 Baseline Model Comparison', fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 15. Cross-Validation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import cross_val_score\n",
    "\n",
    "cv_results = {}\n",
    "for name, model in trained.items():\n",
    "    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')\n",
    "    cv_results[name] = scores\n",
    "    print(f'{name:25s} | CV R²: {scores.mean():.4f} ± {scores.std():.4f}')\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(9, 5))\n",
    "ax.boxplot([cv_results[m] for m in models.keys()],\n",
    "           labels=list(models.keys()), patch_artist=True,\n",
    "           boxprops=dict(facecolor='lightyellow'),\n",
    "           medianprops=dict(color='red', linewidth=2))\n",
    "ax.set_title('🔁 5-Fold Cross-Validation R²', fontsize=12, fontweight='bold')\n",
    "ax.set_ylabel('R² Score')\n",
    "ax.set_ylim(0, 1.1)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 16. Actual vs Predicted Plots"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n",
    "\n",
    "for i, (name, model) in enumerate(trained.items()):\n",
    "    preds = model.predict(X_test)\n",
    "    r2 = r2_score(y_test, preds)\n",
    "    lim = max(y_test.max(), preds.max()) * 1.05\n",
    "\n",
    "    axes[i].scatter(y_test, preds, alpha=0.35, s=20,\n",
    "                    color=PALETTE[i], edgecolors='none')\n",
    "    axes[i].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect fit')\n",
    "    axes[i].set_xlim(0, lim)\n",
    "    axes[i].set_ylim(0, lim)\n",
    "    axes[i].set_title(f'{name}\\nR² = {r2:.4f}', fontsize=10, fontweight='bold')\n",
    "    axes[i].set_xlabel('Actual (T)')\n",
    "    axes[i].set_ylabel('Predicted (T)' if i == 0 else '')\n",
    "    axes[i].legend(fontsize=8)\n",
    "\n",
    "fig.suptitle('🔵 Actual vs Predicted — All Models', fontsize=13, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 17. Residual Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "\n",
    "for i, (name, model) in enumerate(trained.items()):\n",
    "    preds = model.predict(X_test)\n",
    "    residuals = y_test.values - preds\n",
    "\n",
    "    sns.histplot(residuals, kde=True, ax=axes[0][i], color=PALETTE[i], bins=30)\n",
    "    axes[0][i].axvline(0, color='red', linestyle='--', linewidth=1.5)\n",
    "    axes[0][i].set_title(f'{name}\\nResidual Distribution', fontsize=9)\n",
    "\n",
    "    axes[1][i].scatter(preds, residuals, alpha=0.3, s=15, color=PALETTE[i])\n",
    "    axes[1][i].axhline(0, color='red', linestyle='--', linewidth=1.5)\n",
    "    axes[1][i].set_title(f'{name}\\nResiduals vs Predicted', fontsize=9)\n",
    "\n",
    "fig.suptitle('📉 Residual Analysis', fontsize=13, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 18. Feature Importance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "rf = trained['Random Forest']\n",
    "importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=True)\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importances)))\n",
    "importances.plot(kind='barh', ax=axes[0], color=colors_fi, edgecolor='grey', linewidth=0.4)\n",
    "axes[0].set_title('🌟 Random Forest Feature Importances', fontsize=11, fontweight='bold')\n",
    "axes[0].set_xlabel('Importance Score')\n",
    "axes[0].axvline(importances.mean(), color='red', linestyle='--',\n",
    "                label=f'Mean: {importances.mean():.3f}')\n",
    "axes[0].legend()\n",
    "\n",
    "top5 = importances.sort_values(ascending=False).head(5)\n",
    "axes[1].pie(top5.values, labels=top5.index, autopct='%1.1f%%',\n",
    "            colors=PALETTE[:5], startangle=90)\n",
    "axes[1].set_title('Top 5 Features Share', fontsize=11, fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 19. Hyperparameter Tuning (GridSearchCV)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import GridSearchCV\n",
    "\n",
    "print('🔍 Starting GridSearchCV for Random Forest...')\n",
    "\n",
    "param_grid = {\n",
    "    'n_estimators': [50, 100, 200],\n",
    "    'max_depth': [None, 10, 20],\n",
    "    'min_samples_split': [2, 5],\n",
    "    'max_features': ['sqrt', 'log2']\n",
    "}\n",
    "\n",
    "grid_search = GridSearchCV(\n",
    "    RandomForestRegressor(random_state=42),\n",
    "    param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1\n",
    ")\n",
    "grid_search.fit(X_train, y_train)\n",
    "\n",
    "best_rf = grid_search.best_estimator_\n",
    "print(f'\\n✅ Best Parameters: {grid_search.best_params_}')\n",
    "print(f'✅ Best CV R²: {grid_search.best_score_:.4f}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 20. Final Model Evaluation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "trained['RF Tuned'] = best_rf\n",
    "\n",
    "all_results = [evaluate_model(m, n, X_train, y_train, X_test, y_test)\n",
    "               for n, m in trained.items()]\n",
    "final_df = pd.DataFrame(all_results).set_index('Model')\n",
    "\n",
    "print('=== FINAL MODEL COMPARISON ===')\n",
    "display(final_df.style\n",
    "        .background_gradient(cmap='RdYlGn', subset=['R² Test'])\n",
    "        .highlight_min(subset=['RMSE', 'MAE'], color='lightgreen')\n",
    "        .format({'R² Train': '{:.4f}', 'R² Test': '{:.4f}',\n",
    "                 'RMSE': '{:,.0f}', 'MAE': '{:,.0f}'))\n",
    "\n",
    "base_r2 = final_df.loc['Random Forest', 'R² Test']\n",
    "tuned_r2 = final_df.loc['RF Tuned', 'R² Test']\n",
    "print(f'\\n📈 Tuning improvement: {(tuned_r2 - base_r2)*100:+.2f} pp in R² Test')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Final model deep dive\n",
    "best_preds = best_rf.predict(X_test)\n",
    "residuals = y_test.values - best_preds\n",
    "pct_errors = np.abs(residuals) / (y_test.values + 1) * 100\n",
    "\n",
    "fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n",
    "\n",
    "# Actual vs Predicted\n",
    "lim = max(y_test.max(), best_preds.max()) * 1.05\n",
    "axes[0].scatter(y_test, best_preds, alpha=0.4, s=20, color='steelblue')\n",
    "axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5)\n",
    "axes[0].set_title(f'🎯 Actual vs Predicted\\nR²={r2_score(y_test, best_preds):.4f}',\n",
    "                  fontsize=11, fontweight='bold')\n",
    "axes[0].set_xlabel('Actual (T)')\n",
    "axes[0].set_ylabel('Predicted (T)')\n",
    "\n",
    "# Residuals\n",
    "sns.histplot(residuals, kde=True, ax=axes[1], bins=35, color='steelblue')\n",
    "axes[1].axvline(0, color='red', linestyle='--')\n",
    "axes[1].set_title('Residual Distribution', fontsize=11, fontweight='bold')\n",
    "\n",
    "# % Error\n",
    "sns.histplot(pct_errors, kde=True, ax=axes[2], bins=35, color='darkorange')\n",
    "axes[2].axvline(np.median(pct_errors), color='red', linestyle='--',\n",
    "                label=f'Median: {np.median(pct_errors):.1f}%')\n",
    "axes[2].set_title('% Error Distribution', fontsize=11, fontweight='bold')\n",
    "axes[2].legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 21. Error Analysis by Governorate"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_all_scaled = pd.DataFrame(scaler.transform(df[X_COLS].fillna(0)), columns=X_COLS)\n",
    "\n",
    "df_err = df[['Governorate', 'Cereales (T)']].copy().reset_index(drop=True)\n",
    "df_err['Predicted'] = best_rf.predict(X_all_scaled)\n",
    "df_err['AbsError'] = np.abs(df_err['Cereales (T)'] - df_err['Predicted'])\n",
    "\n",
    "gov_error = df_err.groupby('Governorate')['AbsError'].mean().sort_values(ascending=False)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(14, 5))\n",
    "colors = plt.cm.RdYlGn_r(np.linspace(0, 0.8, len(gov_error)))\n",
    "ax.bar(gov_error.index, gov_error.values / 1e3, color=colors, edgecolor='grey', linewidth=0.4)\n",
    "ax.set_title('🗺️ Average Prediction Error by Governorate', fontsize=12, fontweight='bold')\n",
    "ax.set_ylabel('Mean Abs Error (Thousand Tonnes)')\n",
    "plt.xticks(rotation=45, ha='right', fontsize=8)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 22. Conclusions & Recommendations\n",
    "\n",
    "### Model Performance Summary\n",
    "\n",
    "| Model | R² Test | Interpretation |\n",
    "|---|---|---|\n",
    "| Linear Regression | ~0.25 | Too simple — misses non-linear dynamics |\n",
    "| Decision Tree | ~0.95 | Overfitting — memorizes regional splits |\n",
    "| Random Forest (base) | ~0.96 | Strong generalization |\n",
    "| **RF Tuned (best)** | **~0.97** | **Best balance of accuracy & robustness** |\n",
    "\n",
    "### Key Findings\n",
    "\n",
    "1. **Governorate** is the dominant predictor (~28% importance) — it captures soil type, altitude, and microclimate simultaneously.\n",
    "2. **Heat Index** (engineered feature) outperforms raw temperature, confirming combined heat-humidity stress drives yield.\n",
    "3. **Solar radiation** is a stronger meteorological predictor than precipitation, likely due to irrigated agriculture dominance.\n",
    "4. **Linear Regression** cannot model the threshold effects between climate variables and crop output (R²=0.25 vs 0.97).\n",
    "5. Some governorates with high olive/arboriculture share show higher cereal prediction errors due to crop-type confounding.\n",
    "\n",
    "### Recommendations\n",
    "\n",
    "- **Short term:** Deploy Tuned RF as production model. Retrain annually with new harvest data.\n",
    "- **Medium term:** Add monthly weather resolution to capture critical growing-season windows.\n",
    "- **Long term:** Explore LSTM/Prophet for temporal modeling. Include soil type, irrigation data, and NDVI satellite indices.\n",
    "\n",
    "---\n",
    "*🌾 EcoCrop Tunisia — Ferdaws Saidi & Aya Gharsalli — 2024*"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}